# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema URL.


In [ ]:
# Install mlcroissant library if needed
!pip install mlcroissant


## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load Croissant Dataset, getting full metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\nPublished: {metadata.datePublished if hasattr(metadata, 'datePublished') else 'N/A'}\nVersion: {metadata.version if hasattr(metadata, 'version') else 'N/A'}")


## 2. Data Overview
Review the available record sets, fields, and their IDs. All entities are referenced by their `@id` as per Croissant best practices.


In [ ]:
# List all record sets and their fields' @id
record_sets = list(dataset.record_sets())  # These are their @id values
print('Record sets (@id):')
for rs in record_sets:
    rs_info = dataset.record_set(rs)
    print(f"- {rs}")
    # List fields in each record set
    if hasattr(rs_info, 'fields'):
        print(f"  Fields (by @id):")
        for field in rs_info.fields:
            print(f"    - {field['@id']}")
    else:
        print("  [No fields available]")

# Peek some records from one (first) record set
print(f"\nFirst few records from the first record set ({record_sets[0]}):")
for idx, record in enumerate(dataset.records(record_set=record_sets[0])):
    print(record)
    if idx > 2:
        break


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.


In [ ]:
# Extract data from record sets to pandas DataFrames
dfs = {}
for rs in record_sets:
    # Load all records for this record set
    records = list(dataset.records(record_set=rs))
    if len(records) > 0 and isinstance(records[0], dict):
        dfs[rs] = pd.DataFrame(records)

# Print columns for the first record set
main_record_set = record_sets[0]
print(f"Fields in record set {main_record_set} (DataFrame columns):\n{dfs[main_record_set].columns.tolist()}")
dfs[main_record_set].head()


## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping by field `@id`s.


In [ ]:
# Let's choose the GAD-7 total score field, if present, and group by gender
main_df = dfs[main_record_set]

# Find a numeric field (by @id) for demo; let's try 'gad7_total_score' and group by 'gender'
# Replace these @ids with the actual field @ids found in section 2 if different
numeric_field_id = None
group_field_id = None
for col in main_df.columns:
    clow = col.lower()
    if 'gad' in clow and ('score' in clow or 'total' in clow):
        numeric_field_id = col
    if 'gender' in clow:
        group_field_id = col

print(f"Numeric field selected for analysis: {numeric_field_id}")
print(f"Group field selected: {group_field_id}")

# Only proceed if those fields exist
if numeric_field_id and group_field_id:
    # Remove missing/invalid data
    filtered_df = main_df[main_df[numeric_field_id].notnull()]
    threshold = 5   # For demo, filter to those above 5
    filtered_df2 = filtered_df[filtered_df[numeric_field_id] > threshold].copy()

    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df2[[numeric_field_id, group_field_id]].head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df2[norm_col] = (filtered_df2[numeric_field_id] - filtered_df2[numeric_field_id].mean()) / filtered_df2[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df2[[numeric_field_id, norm_col]].head())
    
    # Group by gender and show the average normalized score
    if group_field_id in filtered_df2.columns:
        grouped_df = filtered_df2.groupby(group_field_id)[[numeric_field_id, norm_col]].mean()
        print(f"\nGrouped data by {group_field_id} (mean):")
        print(grouped_df.head())
else:
    print("Could not find suitable numeric or group fields for EDA. Please adjust the @id selection above.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and group_field_id and numeric_field_id in main_df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    plt.figure(figsize=(8,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()
else:
    print("No suitable fields for visualization.")


## 6. Conclusion
This notebook demonstrated how to load, explore, and analyze mental health survey data from Kilifi County, Kenya, using the `mlcroissant` library. With standard field and record set `@id` references, you can flexibly process and adapt the code for future Croissant-based datasets.
